In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,tissue,Stage,mna (cn>5)
samples,,,,
GSM4716627,JB_1,Primary neuroblastoma tumor,s1,no
GSM4716628,JB_2,Primary neuroblastoma tumor,s1,no
GSM4716629,JB_3,Primary neuroblastoma tumor,s1,no
GSM4716630,JB_4,Primary neuroblastoma tumor,s1,no
GSM4716631,JB_5,Primary neuroblastoma tumor,s1,no
...,...,...,...,...
GSM4716719,JB_96,Primary neuroblastoma tumor,s4S,no
GSM4716720,JB_97,Primary neuroblastoma tumor,s4S,no
GSM4716721,JB_98,Primary neuroblastoma tumor,s4S,no


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

,sample_title,tissue,Stage,mna (cn>5),100287102,653635,102466751,107985730,100302278,645520,...,4538,4564,4575,4568,4540,4541,4556,4519,4576,4571
samples,,,,,,,,,,,,,,,,,,,,,
GSM4716627,JB_1,Primary neuroblastoma tumor,s1,no,0,0,0,0,0,0,...,43,393,469,16,74,36,9419,37,5,156
GSM4716629,JB_3,Primary neuroblastoma tumor,s1,no,0,0,0,0,0,0,...,23,15,148,5,36,17,2076,13,0,30
GSM4716630,JB_4,Primary neuroblastoma tumor,s1,no,0,0,0,0,0,0,...,199,86,390,14,202,51,10645,94,2,81
GSM4716631,JB_5,Primary neuroblastoma tumor,s1,no,0,0,0,0,0,0,...,346,550,1944,20,289,76,2570,123,2,217
GSM4716634,JB_8,Primary neuroblastoma tumor,s1,no,0,0,0,0,0,0,...,650,332,830,38,519,119,13775,235,1,127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM4716719,JB_96,Primary neuroblastoma tumor,s4S,no,0,0,0,0,0,0,...,580,657,4991,206,449,88,22697,185,13,2045
GSM4716720,JB_97,Primary neuroblastoma tumor,s4S,no,0,1,0,0,0,0,...,3027,2207,1842,280,1439,356,10507,810,11,1469
GSM4716721,JB_98,Primary neuroblastoma tumor,s4S,no,0,0,0,0,0,0,...,650,1142,4493,21,368,106,12250,232,6,1046


In [3]:
import pandas as pd

# Download supplementary data from: https://www.frontiersin.org/api/v4/articles/647737/file/Table_6.XLSX/647737_supplementary-materials_tables_6_xlsx/1

# Read specific sheet (sheet number 9, which is index 8)
df_clinical_data = pd.read_excel('Table_6.xlsx')
df_clinical_data.to_csv('Table_6_clinical_data.csv', index=False)

# this is to match the sample title in the metadata
df_clinical_data['ID'] = 'JB_' + df_clinical_data['ID'].astype(str)

df_clinical_data
# Merge the dataframes
df_combined = df_clinical_data.merge(
    df_combined,
    left_on='ID',
    right_on='sample_title',
    how='inner'
)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
# drop the ID column
df_combined = df_combined.drop(columns=['ID'])
df_combined.to_csv(data_file_name, index=False)

df_combined

,age at diagnosis,dead or last follow up (days),dead,clinical stage,MYCN amplification,sample_title,tissue,Stage,mna (cn>5),100287102,...,4538,4564,4575,4568,4540,4541,4556,4519,4576,4571
0,356,2121,-,1,-,JB_1,Primary neuroblastoma tumor,s1,no,0,...,43,393,469,16,74,36,9419,37,5,156
1,98,776,-,1,-,JB_3,Primary neuroblastoma tumor,s1,no,0,...,23,15,148,5,36,17,2076,13,0,30
2,298,590,-,1,-,JB_4,Primary neuroblastoma tumor,s1,no,0,...,199,86,390,14,202,51,10645,94,2,81
3,1852,1631,-,1,-,JB_5,Primary neuroblastoma tumor,s1,no,0,...,346,550,1944,20,289,76,2570,123,2,217
4,254,509,-,1,-,JB_8,Primary neuroblastoma tumor,s1,no,0,...,650,332,830,38,519,119,13775,235,1,127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,46,1395,-,4S,-,JB_96,Primary neuroblastoma tumor,s4S,no,0,...,580,657,4991,206,449,88,22697,185,13,2045
75,3,4408,-,4S,-,JB_97,Primary neuroblastoma tumor,s4S,no,0,...,3027,2207,1842,280,1439,356,10507,810,11,1469
76,84,1919,-,4S,-,JB_98,Primary neuroblastoma tumor,s4S,no,0,...,650,1142,4493,21,368,106,12250,232,6,1046
77,73,3478,-,4S,-,JB_99,Primary neuroblastoma tumor,s4S,no,0,...,1865,713,3410,262,1304,308,9464,517,11,1208


# Generate description


In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'Table_6_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
